In [ ]:
import pandas as pd

In [1]:
# Load the combined housing + meeting text dataset into a pandas DataFrame (Change the directory)
df = pd.read_csv('/Users/emilymoore/Downloads/DS 4002 Project 1/combined_housing_market_data.csv')

# Drop redundant/empty columns and rename key
df = df.drop(columns = ["month_year"]) # "month_year" is removed because "month_key" contains the same information
df = df.rename(columns={'month_key': 'date'}) # Rename "month_key" to "date" for clarity and consistency in time-series analysis

# Convert date to datetime objects
df['date'] = pd.to_datetime(df['date']) # Convert the 'date' column to datetime format so it can be used for time-series operations
df['date'] = df['date'].dt.strftime('%Y-%m') # Format the date to show only year-month (e.g., 2025-03), This ensures consistent monthly time indexing

# Convert percentage strings (e.g., "100.00%") into numeric float values
df['median_sold/ask_price_ratio'] = (
    df['median_sold/ask_price_ratio'].str.replace('%', '').astype(float) / 100 
) # Remove the "%" symbol, convert to float, divide by 100 to convert from percent to decimal form (e.g., 1.00)

# Calculate housing price volatility as the month-over-month percentage change
df['price_volatility'] = df['median_sales_price'].pct_change() # pct_change() computes (Price_t - Price_{t-1}) / Price_{t-1}
df['price_volatility'] = df['price_volatility'].fillna(0) # Replace the first NaN (created by pct_change for the first observation) with 0 since there is no prior month to compare

# Fill missing minute mentions with 0 (Assuming NaN means no meeting/mentions)
minute_cols = ['housing_mentions', 'uncertainty_mentions', 'housing_uncertainty_sentences', 'uncertainty_score']
df[minute_cols] = df[minute_cols].fillna(0) # If a value is NaN, assume there was no meeting or no mentions that month

# Define a function to calculate meeting duration in minutes
def get_duration(start, end):
    try:
        fmt = '%I:%M %p' # Define the expected time format (e.g., 12:05 PM)
        tdelta = pd.to_datetime(end, format=fmt) - pd.to_datetime(start, format=fmt) # Convert both start and end times to datetime objects
        # Then compute the difference between them
        return tdelta.total_seconds() / 60 # Convert time difference from seconds to minutes
    except:
        return None # If start or end time is missing/invalid, return None

# Apply the duration function row-by-row to compute meeting length
df['meeting_duration_min'] = df.apply(lambda row: get_duration(row['start_time'], row['end_time']), axis=1) # axis=1 means apply across columns (row-wise)

# Sort chronologically and save
df = df.sort_values('date')

# You will need to change the directory again
df.to_csv('/Users/emilymoore/Downloads/DS 4002 Project 1/cleaned_housing_market_data.csv', index=False) # Save the cleaned dataset to a new CSV file for regression analysis

print(f'Cleaned {df}')

NameError: name 'pd' is not defined